# Part 2 — Graph Laplacian diffusion

## Aim of part 2

- Replace manual neighbour averaging with a mathematically clearer diffusion model.

## Reasoning

- In Part 1, signal propagation between nuclei was modelled using a simple neighbour-averaging rule. This captured the biological intuition that corrected nuclei influence nearby nuclei through spatially localised signalling.

- To make the mechanism mathematically explicit and more realistic spatial arrangements of nuclei, I reformulated signal propagation using the graph Laplacian operator.

- Diffusion on discrete spatial systems is commonly represented using Laplacian operators defined on interaction networks (Masuda, Porter & Lambiotte, 2017). This allows signal transport to be described as a dynamical process evolving over a graph rather than along a fixed linear array.

For this project, the modelling chain taken from my MSc thesis:

```text
→ local cross-correction appears spatial
→ represent nuclei as interacting nodes
→ model signal transport as diffusion over a graph
→ use the graph Laplacian to formalise that diffusion
```

## Representing nuclei as a spatial interaction graph

Instead of treating nuclei only as positions in a Python list, I now represent them as nodes in a graph. Edges describe which nuclei can exchange signal.

For Part 2, I still use a simple line graph because it keeps the geometry close to Part 1.

This gives a clean transition:

| Part | Spatial representation | Transport rule |
|---|---|---|
| Part 1 | 1D list of nuclei | manual neighbour averaging |
| Part 2 | 1D graph of nuclei | graph Laplacian diffusion |

I changed this so that the connections are stored in an adjacency matrix `A`, instead of being manually set using the `left` and `right` variables.


In [1]:
import numpy as np

def line_adjacency(n: int) -> np.ndarray:
    A = np.zeros((n, n), dtype=float)
    for i in range(n - 1):
        A[i, i + 1] = 1.0
        A[i + 1, i] = 1.0
    return A

def simulate_laplacian_line(
    n_nodes: int = 20,
    corrected_fraction: float = 0.1,
    steps: int = 100,
    alpha: float = 0.4,
    beta: float = 0.05,
    dt: float = 0.1,
    threshold: float = 1.0,
):
    A = line_adjacency(n_nodes)
    D = np.diag(A.sum(axis=1))
    L = D - A

    u = np.zeros(n_nodes, dtype=float)
    rescued = np.zeros(n_nodes, dtype=bool)

    n_corrected = max(1, round(n_nodes * corrected_fraction))
    corrected = np.zeros(n_nodes, dtype=bool)
    corrected[:n_corrected] = True
    q = corrected.astype(float) * 0.5

    for _ in range(steps):
        du = -alpha * (L @ u) - beta * u + q
        u = np.maximum(u + dt * du, 0.0)
        rescued = np.logical_or(rescued, u >= threshold)

    return {"signal": u, "rescued": rescued, "adjacency": A, "laplacian": L}

## Using the graph Laplacian?

The graph Laplacian is defined as:

$$
L = D - A
$$

where:

- **A** is the adjacency matrix  
- **D** is the degree matrix  
- **L** is the Laplacian matrix

The intuition is simple: the Laplacian compares each node with its neighbours. If a nucleus has more signal than its neighbours, the diffusion term pushes signal outward. If it has less signal than its neighbours, the diffusion term pulls signal inward.

So the Laplacian encodes the idea:

```text
signal moves from high-concentration nodes toward lower-concentration neighbours
```

## Building the Laplacian matrix

The degree matrix `D` records how many neighbours each nucleus has.  
In a line graph, nuclei in the middle have degree 2, while nuclei at the ends have degree 1.

The Laplacian matrix is then defined as:

$$
L = D - A
$$

where:

- **A** is the adjacency matrix
- **D** is the degree matrix
- **L** is the Laplacian matrix

This matrix acts directly on the signal vector `u`:

```python
L @ u
```

That expression replaces the hand-written left/right mixing from Part 1.

In [2]:
def graph_laplacian(A: np.ndarray) -> np.ndarray:
    """Construct the combinatorial graph Laplacian L = D - A."""
    D = np.diag(A.sum(axis=1))
    return D - A


## Diffusion with source and decay terms

Corrected nuclei are still treated as local signal sources. This is represented by the source vector $q$.

The Part 2 update equation is:

$$
u_{t+1} = u_t + dt\left[-\alpha L u_t - \beta u_t + q\right]
$$

where:

- $u_t$ is the signal vector at timestep $t$
- $-\alpha L u_t$ spreads signal across connected nuclei through graph diffusion
- $-\beta u_t$ models explicit signal decay over time
- $q$ adds continuous signal production from corrected nuclei
- $dt$ controls the size of each numerical update step

This makes the model:

```text
transport + decay + production

In [3]:
def simulate_laplacian_line(
    n_nodes: int = 20,
    corrected_fraction: float = 0.1,
    steps: int = 100,
    alpha: float = 0.4,
    beta: float = 0.05,
    dt: float = 0.1,
    threshold: float = 1.0,
):
    """Simulate graph Laplacian diffusion on a line of nuclei.

    Parameters
    ----------
    n_nodes:
        Number of nuclei represented in the line graph.
    corrected_fraction:
        Fraction of nuclei treated as corrected signal sources.
    steps:
        Number of simulation steps.
    alpha:
        Transport strength. Larger values spread signal faster.
    beta:
        Signal decay rate. Larger values remove signal faster.
    dt:
        Time-step size for the explicit update.
    threshold:
        Signal level required for a nucleus to be marked as rescued.
    """
    A = line_adjacency(n_nodes)
    L = graph_laplacian(A)

    u = np.zeros(n_nodes, dtype=float)
    rescued = np.zeros(n_nodes, dtype=bool)

    n_corrected = max(1, round(n_nodes * corrected_fraction))
    corrected = np.zeros(n_nodes, dtype=bool)
    corrected[:n_corrected] = True
    q = corrected.astype(float) * 0.5

    history = [u.copy()]

    for _ in range(steps):
        du = -alpha * (L @ u) - beta * u + q
        u = np.maximum(u + dt * du, 0.0)
        rescued = np.logical_or(rescued, u >= threshold)
        history.append(u.copy())

    return {
        "signal": u,
        "rescued": rescued,
        "corrected": corrected,
        "adjacency": A,
        "laplacian": L,
        "history": np.array(history),
    }


In [4]:
result = simulate_laplacian_line()
result["signal"], result["rescued"]


(array([2.90503787e+00, 2.41127842e+00, 1.32119800e+00, 6.79233180e-01,
        3.26081940e-01, 1.45700829e-01, 6.04754204e-02, 2.32981406e-02,
        8.33141299e-03, 2.76750070e-03, 8.54963190e-04, 2.46008716e-04,
        6.60436405e-05, 1.65716285e-05, 3.89352918e-06, 8.58125310e-07,
        1.77727485e-07, 3.46530962e-08, 6.39715213e-09, 1.28695980e-09]),
 array([ True,  True,  True, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False]))

## Interpretation of the Part 2 transition

- Part 1 demonstrated that local signal exchange could be simulated using simple neighbour averaging.
- Part 2 reformulates that same idea as diffusion over a graph, making the transport mechanism explicit.
- The adjacency matrix `A` stores which nuclei interact.
- The Laplacian matrix `L = D − A` converts that connectivity into a diffusion operator.
- The update equation now separates transport, decay, and production, which makes later parameter experiments easier to justify.

The conceptual use of Masuda, Porter and Lambiotte (2017) is therefore narrow and specific: the paper supports the decision to treat local signal spread as diffusion on a network. The biological hypothesis still comes from my MSc thesis, while the graph Laplacian provides the mathematical language for implementing that hypothesis computationally.
